[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/UrbanAccessAnalyzer/blob/main/examples/transit_level_of_service_cambridge.ipynb)

# 🚌 Public Transport Level of Service -- Cambridge, MA

This notebook measures **transit level of service (LOS)**: how good is
walk-to-transit access, street by street, across a city? It's inspired by
(and reuses the same city as) `transitLOS`'s own
`transit_level_of_service_walkthrough.ipynb`, adapted here to also show off
`geohierarchy` multi-resolution mapping and `pycensus` population, and to
add a very simple population-weighted equity readout.

**What "level of service" means here:** every street segment gets an
`access_score` in `[0, 1]`, combining (a) how close the segment is to the
nearest transit stop and (b) how good that stop is (frequency, number of
routes, etc. -- see `transitlos.stop_scores.compute_stop_scores`). A
segment right next to a frequent, well-connected stop scores near 1; a
segment far from any stop, or only near an infrequent one, scores near 0.

**Why Cambridge, MA:** it has real, richly-served GTFS feeds available from
the Mobility Database (MBTA bus/subway plus several regional operators),
and it's the same place used throughout this repo family's other examples
(`UrbanAccessAnalyzer/examples/walkability.ipynb`,
`pyGTFSHandler/examples/cambridge_massachusetts_usa_example.ipynb`), so
results here are directly comparable to those notebooks.

In [ ]:
# If using colab
# Takes around 3-5 min (installs several packages + geo libraries)

# !pip install "UrbanAccessAnalyzer[plot] @ git+https://github.com/CityScope/UrbanAccessAnalyzer.git@v2.0.0"
# !pip install "geohierarchy @ git+https://github.com/CityScope/geohierarchy.git"
# !pip install "pycensus @ git+https://github.com/CityScope/pyCensus.git"
# !pip install "pyGTFSHandler @ git+https://github.com/CityScope/pyGTFSHandler.git"
# transitlos isn't published/pushed to GitHub yet -- this will start working
# once it is. Until then, run this notebook from a machine with a local
# checkout of transitLOS on the Python path (e.g. `pip install -e ../../transitLOS`
# from this repo's examples/ folder) instead of this line.
# !pip install "transitlos @ git+https://github.com/CityScope/transitLOS.git"

# Restart notebook after installing this if needed

In [ ]:
import warnings

import matplotlib.pyplot as plt
import geopandas as gpd

import transitlos
from transitlos import compute_level_of_service
from UrbanAccessAnalyzer.api import AreaOfInterest

warnings.filterwarnings("ignore")  # geographic-CRS / centroid warnings from intermediate steps

## 1. Define the area of interest

`transitlos.compute_level_of_service` takes an `AreaOfInterest` (from
`UrbanAccessAnalyzer.api`), which just wraps a polygon GeoDataFrame. We
build one for Cambridge, MA via `pyGTFSHandler.utils.geocoding.get_city_geometry`,
the same helper used to build the AOI and download the GTFS feed below, so
they cover exactly the same place.

In [ ]:
from pyGTFSHandler.utils.geocoding import get_city_geometry

city_name = "Cambridge, Massachusetts, USA"
city_gdf = get_city_geometry(city_name)  # single-row polygon GeoDataFrame
aoi = AreaOfInterest(city_gdf)

city_gdf.plot(edgecolor="black", facecolor="none")
plt.title(city_name)
plt.axis("off")

## 2. Get a real GTFS feed

`feed_source` is passed straight through to `pyGTFSHandler.feed.Feed`, so
any path/URL/list of GTFS zip or directory paths it understands works.
Here we search and download a real feed for Cambridge from the
[Mobility Database](https://mobilitydatabase.org/), which requires a free
API refresh token.

**Getting the token:** sign up at https://mobilitydatabase.org/ for a free
refresh token, then either set it as the `MOBILITY_DATABASE_REFRESH_TOKEN`
environment variable, add it to `api_keys.json` under `"mobility_database"`
(see `api_keys.example.json` in the pyGTFSHandler repo root), or paste it
below when prompted. `MobilityDatabaseDownloader` checks, in order: an
explicit argument, the environment variable, then `api_keys.json` -- so
passing `None`/leaving the token blank tries all three before this cell
prompts you interactively.

In [ ]:
import os
import getpass

from pyGTFSHandler.downloaders.mobility_database import MobilityDatabaseDownloader

try:
    # Empty string first: MobilityDatabaseDownloader still checks the env var
    # and api_keys.json before giving up (see pyGTFSHandler.downloaders.base).
    downloader = MobilityDatabaseDownloader("")
except ValueError:
    # Automatic resolution failed -- ask for it directly instead.
    token = os.environ.get("MOBILITY_DATABASE_REFRESH_TOKEN") or getpass.getpass(
        "Enter your Mobility Database refresh token (https://mobilitydatabase.org/): "
    )
    downloader = MobilityDatabaseDownloader(token)

In [ ]:
import glob

from pyGTFSHandler.utils.geocoding import get_geographic_suggestions_from_string

geo_suggestions = get_geographic_suggestions_from_string(city_name)

feeds = downloader.search_feeds(
    country_code=geo_suggestions["country_codes"],
    subdivision_name=geo_suggestions["subdivision_names"],
    municipality=geo_suggestions["municipalities"],
    is_official=True,
)
orig_file_paths = downloader.download_feeds(
    feeds=feeds,
    download_folder="outputs/orig_gtfs_files",
    overwrite=False,
)

feed_source = sorted(glob.glob("outputs/orig_gtfs_files/*"))
feed_source

## 3. Compute the transit level-of-service layer

`transitlos.compute_level_of_service` is the "do everything" entry point:
it builds (or loads a cached) walking street network, loads and scores the
downloaded GTFS stops, and runs an `UrbanAccessAnalyzer.api.AccessibilityAnalyzer`
to produce a street-edges GeoDataFrame with an `access_score` column.

Passing an explicit `date_range` (a known-good service day for these
feeds) skips an expensive auto-detection step -- see the function's
docstring (`transitlos.level_of_service.compute_level_of_service`) for
details. If you're running this notebook well after the date below, GTFS
feeds may no longer cover it; drop `date_range` entirely to fall back to
automatic detection (slower, but always works against whatever's
downloaded).

In [ ]:
from datetime import date

analysis_date = date(2026, 8, 21)

access_gdf = compute_level_of_service(
    aoi,
    feed_source=feed_source,
    region="north_america",
    cache_dir="./_network_cache",
    pbf_path="./us_massachusetts.osm.pbf",
    walk_distance_steps=(400, 800, 1200),
    date_range=(analysis_date, analysis_date),
)
print(f"{len(access_gdf)} scored street segments")
access_gdf["access_score"].describe()

## 4. Population: `pycensus.worldwide.worldpop`

We use `pycensus.worldwide.worldpop` for population, not a country-specific
source like the US Census -- keeping this notebook (like the other
`UrbanAccessAnalyzer` examples) reusable for transit-LOS analysis anywhere
in the world, not just the US. `download_worldpop_population` fetches a
100m-resolution gridded population raster from the WorldPop Hub (no API
key required).

In [ ]:
import pycensus.worldwide.worldpop as worldpop

population_file = worldpop.download_worldpop_population(
    aoi.gdf,
    2025,
    folder="worldpop_cache",
    resolution="100m",
)

## 5. Combine into a `geohierarchy.GeoHierarchy` and map the result

An H3-cell grid (`geohierarchy.utils.h3_cells`) covering Cambridge is the
hierarchy's single level. Two external sources are layered onto it:

- The transit `access_score` street edges, via `add_vector_data` --
  `Mean()`-aggregated (average a 0-1 score, don't sum it) with
  `geoweight_by="length"` since streets are lines.
- The WorldPop population raster, via `add_raster_data` --
  `Sum()`-aggregated (population counts add up).

The joined result renders as an interactive, multi-resolution
`geohierarchy.maps.folium.HierarchyMap` -- pan/zoom, and click a cell to
see its population and mean transit access score.

In [ ]:
from geohierarchy import GeoHierarchy, Sum, Mean
from geohierarchy.maps.folium import HierarchyMap, ColorSpec
from geohierarchy.utils import h3_cells

h3_resolution = 9  # ~0.1 km2 hexagons -- fine enough for a city-scale AOI
h3_grid = h3_cells(aoi.gdf, resolution=h3_resolution)

hierarchy = GeoHierarchy()
hierarchy.add_level("h3", h3_grid, id_col="h3")
hierarchy.add_vector_data(
    access_gdf,
    level="h3",
    columns=["access_score"],
    agg=Mean(),
    geoweight_by="length",
)
hierarchy.add_raster_data(population_file, column="population", level="h3", agg=Sum())

results_h3 = hierarchy["h3"]
results_h3 = results_h3[results_h3.intersects(aoi.gdf.union_all())]
results_h3.head()

In [ ]:
access_style = ColorSpec(column="access_score", cmap="viridis", domain=(0, 1))

los_map = HierarchyMap(hierarchy, levels=["h3"], tiles_dir="transit_los_map_tiles")
los_map.configure_level(
    "h3",
    style=access_style,
    popup_fields=["population", "access_score"],
)
los_map.build()
los_map.save("transit_los_map.html")

`HierarchyMap` serves geometry as vector tiles fetched over plain HTTP, so
opening the saved HTML file directly (double-click / `file://`) will show
the basemap but silently fail to load geometry -- browsers block local
`fetch()` calls from a `file://` page. The cell below starts a tiny local
server so the map previews correctly inside this notebook.

In [ ]:
import functools
import http.server
import threading

from IPython.display import IFrame

PORT = 8001
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=".")
httpd = http.server.ThreadingHTTPServer(("localhost", PORT), handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()

IFrame(f"http://localhost:{PORT}/transit_los_map.html", width="100%", height=600)

## 6. A simple equity readout

`transitLOS` has a dedicated `transitlos.equity` module for rigorous
population-weighted equity statistics (weighted mean/median, Gini
coefficient of access, ...) -- see `equity_analysis_walkthrough.ipynb` in
`transitLOS/examples/` if you want the full version. Here we keep it
deliberately simple, computed directly from `results_h3` (population +
mean `access_score` per H3 cell):

- **Population-weighted mean access score**: the average `access_score`
  across the city, weighting each cell by how many people live there --
  answers "how good is transit access for the *average resident*?", as
  opposed to a plain unweighted average across cells (which would treat an
  empty cell the same as a dense one).
- **Share of the population "well served"**: the fraction of residents
  living in a cell with `access_score >= 0.5`, a simple well-served/
  underserved split.

In [ ]:
import numpy as np

weights = results_h3["population"].to_numpy()
scores = results_h3["access_score"].to_numpy()
mask = ~np.isnan(scores) & (weights > 0)

weighted_mean_access = np.average(scores[mask], weights=weights[mask])
share_well_served = weights[mask & (scores >= 0.5)].sum() / weights[mask].sum()

print(f"Population-weighted mean transit access score: {weighted_mean_access:.2f} (0-1 scale)")
print(f"Share of residents in a well-served area (access_score >= 0.5): {share_well_served:.0%}")

## Next steps

- Swap in a different `analysis_date` / feed set to compare transit LOS
  across seasons or after a service change.
- Use `transitlos.equity.equity_statistics` directly (on a table produced
  by `transitlos.population.cross_with_population`) for the fuller equity
  toolkit (weighted median, Gini coefficient, breakdowns by group), as
  shown in `transitLOS/examples/equity_analysis_walkthrough.ipynb`.
- Compare this transit-access layer against the walkability/green-space
  layers in the other `UrbanAccessAnalyzer` example notebooks for the same
  city, to see where good walk access and good transit access do (or
  don't) overlap.